<a href="https://colab.research.google.com/github/lucas6028/x-coach/blob/main/notebooks/run_rehab24_videomae_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REHAB24-6 × VideoMAE 特徵提取 (Colab)

在 Colab GPU 上對 **REHAB24-6** 每個動作重複 (repetition) 提取 VideoMAE 特徵。

流程：
1. 確認 GPU
2. 掛載 Google Drive（**原始碼已上傳到 Drive**）
3. 直接從 [Zenodo](https://zenodo.org/records/13305826) 下載 `videos.zip` + `Segmentation.csv`
4. 解壓縮並偵測資料夾結構
5. 建立 manifest / splits / labels (`scripts/rehab24/build_manifest.py`)
6. 跑 VideoMAE 特徵提取 (`scripts/rehab24/extract_videomae_features.py`)
7. 打包特徵存回 Drive

> VideoMAE 只需要影片 (`videos.zip`) 與切割標註 (`Segmentation.csv`)，因此 manifest 用 `--skip-path-validation` 略過尚未下載的 skeleton `.npy`。
> 之後若要跑 skeleton / fusion，再回 Zenodo 下載 `2d_joints.zip` / `3d_joints.zip` 即可。

### 0. 確認 GPU（Runtime → Change runtime type → GPU）

In [1]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU 0: Tesla T4 (UUID: GPU-ecec9b12-7b6a-1d25-f570-869364669f64)
CUDA available: True
Device: Tesla T4


### 1. 掛載 Google Drive（原始碼）

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. 設定路徑

把 `REPO_ROOT` 改成你在 Drive 上放原始碼的位置（裡面要有 `scripts/` 和 `src/`）。

In [3]:
import os

# === 原始碼位置（已上傳到 Google Drive，請改成你的路徑）===
REPO_ROOT = "/content/drive/MyDrive/x-coach"

# === 下載與解壓縮位置（Colab 本機磁碟，讀寫快）===
DOWNLOAD_DIR = "/content/REHAB24-6"
DATA_ROOT = DOWNLOAD_DIR  # 解壓後第 4 步會自動偵測並更新
SEG_PATH = os.path.join(DOWNLOAD_DIR, "Segmentation.csv")

# === manifest / splits / labels 存回 Drive，可長期保存 ===
PROCESSED_ROOT = "/content/drive/MyDrive/x-coach/data/REHAB24-6/processed"

# === VideoMAE 特徵先寫到本機，最後再打包回 Drive ===
LOCAL_FEATURE_DIR = "/content/videomae_features"
DRIVE_FEATURE_ZIP = "/content/drive/MyDrive/x-coach/data/REHAB24-6/videomae_features.zip"

# 兩個相機視角；只要單一視角可改成 "cam17"
CAMERAS = "cam17,cam18"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(PROCESSED_ROOT, exist_ok=True)
os.makedirs(LOCAL_FEATURE_DIR, exist_ok=True)
assert os.path.isdir(os.path.join(REPO_ROOT, "scripts", "rehab24")), \
    f"在 {REPO_ROOT} 找不到 scripts/rehab24，請確認 REPO_ROOT 設定正確"
print("REPO_ROOT     :", REPO_ROOT)
print("DOWNLOAD_DIR  :", DOWNLOAD_DIR)
print("PROCESSED_ROOT:", PROCESSED_ROOT)

REPO_ROOT     : /content/drive/MyDrive/x-coach
DOWNLOAD_DIR  : /content/REHAB24-6
PROCESSED_ROOT: /content/drive/MyDrive/x-coach/data/REHAB24-6/processed


### 3. 從 Zenodo 下載資料集

`videos.zip` 約 **2.7 GB**，`Segmentation.csv` 約 53 KB。`wget -c` 可續傳，重跑不會重新下載。

In [4]:
!cp -r "{REPO_ROOT}/data/REHAB24-6/videos.zip" "{DOWNLOAD_DIR}/videos.zip"
# !wget -c -O "{DOWNLOAD_DIR}/videos.zip" "https://zenodo.org/records/13305826/files/videos.zip?download=1"
!wget -c -O "{SEG_PATH}" "https://zenodo.org/records/13305826/files/Segmentation.csv?download=1"
!ls -lh "{DOWNLOAD_DIR}"

--2026-06-02 02:18:00--  https://zenodo.org/records/13305826/files/Segmentation.csv?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 188.185.48.75, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53290 (52K) [text/plain]
Saving to: ‘/content/REHAB24-6/Segmentation.csv’

/content/REHAB24-6/ 100%[===================>]  52.04K  --.-KB/s    in 0.05s   

2026-06-02 02:18:00 (1003 KB/s) - ‘/content/REHAB24-6/Segmentation.csv’ saved [53290/53290]

total 2.5G
-rw-r--r-- 1 root root  53K Jun  2 02:18 Segmentation.csv
-rw------- 1 root root 2.5G Jun  2 02:18 videos.zip


### 4. 解壓縮並偵測資料夾結構

manifest 內的影片路徑形如 `ExN/<video_id>-Camera17-30fps.mp4`，所以 `--data-root` 必須是 `ExN` 資料夾的**上一層**；下面會自動偵測。

In [5]:
!unzip -q -o "{REPO_ROOT}/data/REHAB24-6/videos.zip" -d "{DOWNLOAD_DIR}"

In [6]:
from pathlib import Path

cam17 = list(Path(DOWNLOAD_DIR).rglob("*-Camera17-30fps.mp4"))
cam18 = list(Path(DOWNLOAD_DIR).rglob("*-Camera18-30fps-transposed.mp4"))
print(f"找到 {len(cam17)} 支 Camera17、{len(cam18)} 支 Camera18 影片")
assert cam17, "找不到影片，請確認 videos.zip 已正確解壓縮"

# data-root = ExN 資料夾的上一層
DATA_ROOT = str(cam17[0].parent.parent)
print("偵測到的 DATA_ROOT:", DATA_ROOT)
print("範例影片:", cam17[0])

找到 65 支 Camera17、65 支 Camera18 影片
偵測到的 DATA_ROOT: /content/REHAB24-6
範例影片: /content/REHAB24-6/Ex2/PM_026-Camera17-30fps.mp4


### 5. 建立 manifest / splits / labels

依 person id 做固定切割：train `{1,2,3,4,5,7,10}` · val `{6}` · test `{8,9}`。每個重複 × 相機 = 一個 sample。

In [7]:
!python "{REPO_ROOT}/scripts/rehab24/build_manifest.py" \
  --data-root "{DATA_ROOT}" \
  --segmentation "{SEG_PATH}" \
  --processed-root "{PROCESSED_ROOT}" \
  --cameras "{CAMERAS}" \
  --skip-path-validation

Wrote 2144 REHAB24-6 samples to /content/drive/MyDrive/x-coach/data/REHAB24-6/processed/manifest.csv
train: 1434
val: 212
test: 498


驗證 manifest 引用的影片都存在（跑特徵提取前先確認，避免長時間跑到一半才報錯）。

In [8]:
import sys
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from pathlib import Path
from src.rehab24.dataset import load_manifest, resolve_data_path

rows = load_manifest(Path(PROCESSED_ROOT) / "manifest.csv")
missing = [r["sample_id"] for r in rows
           if not resolve_data_path(Path(DATA_ROOT), r["video_path"]).exists()]
print(f"manifest 共 {len(rows)} 列，缺少影片 {len(missing)} 個")
for s in missing[:10]:
    print("  missing:", s)
assert not missing, "有影片找不到——若只想用單一視角，請把 CAMERAS 改成 'cam17' 後重跑第 5 步"

manifest 共 2144 列，缺少影片 0 個


### 6. 安裝套件

Colab 已內建 `torch` 與 `opencv`，只需補裝 transformers 相關套件。

In [9]:
!pip install -q transformers accelerate timm

### 7. 煙霧測試 (smoke test)

先用 `--limit 4` 確認模型載入、影片讀取、寫檔都正常，再跑完整資料集。

In [ ]:
!python "{REPO_ROOT}/scripts/rehab24/extract_videomae_features.py" \
  --data-root "{DATA_ROOT}" \
  --manifest "{PROCESSED_ROOT}/manifest.csv" \
  --output-dir "{LOCAL_FEATURE_DIR}" \
  --num-clips 4 \
  --device cuda \
  --limit 4

### 8. 完整特徵提取

輸出寫到本機 `LOCAL_FEATURE_DIR/{split}/{sample_id}.npz`。已存在的檔會跳過（加 `--overwrite` 可強制重算），所以 Colab 斷線後可直接重跑接續。

In [20]:
!cp "{REPO_ROOT}/data/REHAB24-6/videomae_features.zip" "/content/videomae_features.zip"
!unzip -q -o "/content/videomae_features.zip" -d "{LOCAL_FEATURE_DIR}"
!ls -lh "{LOCAL_FEATURE_DIR}"

total 12K
drwxr-xr-x 2 root root 12K Jun  1 15:20 train


In [21]:
!python "{REPO_ROOT}/scripts/rehab24/extract_videomae_features.py" \
  --data-root "{DATA_ROOT}" \
  --manifest "{PROCESSED_ROOT}/manifest.csv" \
  --output-dir "{LOCAL_FEATURE_DIR}" \
  --num-clips 4 \
  --device cuda

Loading VideoMAE model `MCG-NJU/videomae-base-finetuned-kinetics` on cuda...
preprocessor_config.json: 100% 271/271 [00:00<00:00, 926kB/s]
config.json: 100% 22.9k/22.9k [00:00<00:00, 11.4MB/s]
model.safetensors: 100% 346M/346M [00:08<00:00, 38.7MB/s]
Loading weights: 100% 182/182 [00:00<00:00, 980.31it/s, Materializing param=encoder.layer.11.output.dense.weight]
VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
fc_norm.bias      | UNEXPECTED |  | 
fc_norm.weight    | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processed 200 manifest rows...
Processed 250 manifest rows...
Processed 300 manifest rows...
Processed 350 manifest rows...
Processed 400 manifest rows...
Processed 450 manifest rows...
Processed 500 manifest rows...
Pr

### 9. 打包並存回 Google Drive

幾千個小 `.npz` 直接寫 Drive 很慢，所以前面寫本機、這裡再壓成一個 zip 存回 Drive。

In [22]:
import os
from pathlib import Path
n = len(list(Path(LOCAL_FEATURE_DIR).rglob("*.npz")))
print(f"共產生 {n} 個特徵檔")
os.makedirs(os.path.dirname(DRIVE_FEATURE_ZIP), exist_ok=True)

共產生 2144 個特徵檔


In [23]:
!cd "{LOCAL_FEATURE_DIR}" && zip -r -q "{DRIVE_FEATURE_ZIP}" .
!ls -lh "{DRIVE_FEATURE_ZIP}"

-rw------- 1 root root 32M Jun  2 06:55 /content/drive/MyDrive/x-coach/data/REHAB24-6/videomae_features.zip


### 下一步

- 解壓 `videomae_features.zip` 後，可接 `scripts/rehab24/fuse_features.py`（需先有 skeleton 特徵）或直接餵 `scripts/rehab24/train_correctness_classifier.py`：
  ```bash
  python scripts/rehab24/train_correctness_classifier.py \
    --feature-dir videomae_features --manifest manifest.csv \
    --train-keys splits/train_keys.json --val-keys splits/val_keys.json \
    --test-keys splits/test_keys.json --labels labels/correctness.json --device cuda
  ```
- `manifest.csv`、`splits/`、`labels/` 已存在 `PROCESSED_ROOT`（Drive），不需重建。